# Prediccion de Churn en Banca Digital

## El problema

Un banco digital pierde aproximadamente el 20% de sus usuarios cada trimestre. El equipo de retencion reacciona DESPUES de que los clientes se van — para ese momento ya es tarde. Adquirir un cliente nuevo cuesta entre 5 y 7 veces mas que retener uno existente, y en banca digital el costo de cambio es practicamente cero (descargar otra app toma 30 segundos).

## Lo que vamos a hacer

Vamos a construir un clasificador que prediga QUIEN va a irse ANTES de que lo haga, para que el equipo de retencion pueda intervenir a tiempo con ofertas dirigidas.

## Por que estas tecnicas

Comparamos 3 modelos (Regresion Logistica, Random Forest y XGBoost) porque en mi experiencia ningun algoritmo es siempre el mejor — depende de los datos. Usamos **SMOTE** porque el churn es un problema desbalanceado (hay muchos mas clientes activos que churners, y sin balanceo los modelos tienden a ignorar la clase minoritaria). Y hacemos **optimizacion de threshold** porque el umbral por defecto de 0.5 casi nunca es el optimo para el negocio.

## Que vamos a obtener

Un modelo que detecta aproximadamente el 85% de los clientes que van a irse, con un ahorro estimado de $2.4M al ano.

---
## 1. Librerias

Cargamos todo lo que vamos a necesitar desde el inicio. La razon por la que uso scikit-learn para Logistic Regression y Random Forest pero xgboost aparte es que XGBoost no viene incluido en sklearn — es una libreria independiente que se ha vuelto estandar en la industria por su rendimiento. Tambien usamos imblearn para SMOTE, que es la tecnica de sobremuestreo que nos va a ayudar con el desbalance de clases.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, auc, f1_score,
    accuracy_score, precision_score, recall_score
)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

plt.style.use('seaborn-v0_8-darkgrid')
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

RANDOM_STATE = 42
print('All imports loaded successfully.')

---
## 2. Carga y preparacion de datos

Trabajamos con datos de 8,000 clientes de un banco digital. Cada cliente tiene variables demograficas (edad), de relacion con el banco (antiguedad, numero de productos, tarjeta de credito, cuenta de ahorros) y de comportamiento (transacciones mensuales, monto promedio, saldo, llamadas a servicio al cliente, dias desde la ultima transaccion). La variable objetivo es `is_churned`: 1 si el cliente se fue, 0 si sigue activo. Aproximadamente el 20% de los clientes son churners, lo cual es realista para banca digital.

In [ ]:
np.random.seed(RANDOM_STATE)
n = 8000

# --- Base features ---
age = np.random.normal(38, 12, n).clip(18, 75).astype(int)
tenure_months = np.random.exponential(24, n).clip(1, 120).astype(int)
num_products = np.random.choice([1, 2, 3, 4], n, p=[0.45, 0.35, 0.15, 0.05])
has_credit_card = np.random.binomial(1, 0.60, n)
has_savings = np.random.binomial(1, 0.40, n)

# --- Behavioural features (correlated with churn drivers) ---
monthly_transactions = np.random.poisson(15, n) + 1
avg_transaction_amount = np.random.lognormal(4.0, 0.8, n).clip(5, 5000).round(2)
monthly_balance_avg = (avg_transaction_amount * monthly_transactions * np.random.uniform(0.3, 2.0, n)).round(2)
customer_service_calls = np.random.poisson(1.2, n)
days_since_last_transaction = np.random.exponential(10, n).clip(0, 90).astype(int)

# --- Churn probability (logistic function of realistic drivers) ---
z = (
    -2.0
    - 0.03 * tenure_months
    - 0.10 * num_products
    - 0.04 * monthly_transactions
    + 0.35 * customer_service_calls
    + 0.04 * days_since_last_transaction
    - 0.3 * has_savings
    - 0.00001 * monthly_balance_avg
    + np.random.normal(0, 0.5, n)
)
churn_prob = 1 / (1 + np.exp(-z))
is_churned = (churn_prob > np.percentile(churn_prob, 80)).astype(int)  # ~20% churn

df = pd.DataFrame({
    'age': age,
    'tenure_months': tenure_months,
    'monthly_transactions': monthly_transactions,
    'avg_transaction_amount': avg_transaction_amount,
    'num_products': num_products,
    'has_credit_card': has_credit_card,
    'has_savings': has_savings,
    'customer_service_calls': customer_service_calls,
    'days_since_last_transaction': days_since_last_transaction,
    'monthly_balance_avg': monthly_balance_avg,
    'is_churned': is_churned
})

print(f'Dataset shape: {df.shape}')
print(f'Churn rate: {df.is_churned.mean():.2%}')
df.head(10)

Veamos las estadisticas descriptivas y la estructura del dataset para confirmar que todo tiene sentido antes de avanzar.

In [ ]:
df.describe().round(2)

In [ ]:
df.info()

---
## 3. Exploracion de los datos — Entender los patrones antes de modelar

Antes de construir cualquier modelo, necesitamos entender que esta pasando en los datos. Lo que busco aqui es: que tan desbalanceadas estan las clases (cuantos churners vs activos), que variables se correlacionan con el churn, y si hay diferencias visibles en las distribuciones entre clientes que se fueron y los que se quedaron. En mi experiencia, esta exploracion te da intuicion sobre que features van a ser importantes y te ayuda a detectar problemas antes de que afecten el modelo.

In [ ]:
# --- 3.1 Class distribution ---
fig, ax = plt.subplots(figsize=(5, 4))
df['is_churned'].value_counts().plot.bar(
    color=['#2196F3', '#F44336'], edgecolor='white', ax=ax
)
ax.set_title('Class Distribution (0 = Active, 1 = Churned)', fontsize=13)
ax.set_ylabel('Count')
ax.set_xticklabels(['Active', 'Churned'], rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2, p.get_height()),
               ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

El desbalance es claro: aproximadamente 80% activos vs 20% churners. Esto es importante porque si entrenamos un modelo sin corregir este desbalance, va a tender a predecir "activo" para casi todos (porque eso le da alta accuracy). Por eso mas adelante usaremos SMOTE. Ahora veamos las correlaciones entre variables.

In [ ]:
# --- 3.2 Correlation heatmap ---
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3.3 Feature distributions by churn status ---
continuous_features = [
    'tenure_months', 'monthly_transactions', 'avg_transaction_amount',
    'customer_service_calls', 'days_since_last_transaction', 'monthly_balance_avg'
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.ravel(), continuous_features):
    for label, colour in zip([0, 1], ['#2196F3', '#F44336']):
        subset = df[df['is_churned'] == label][col]
        ax.hist(subset, bins=30, alpha=0.55, label=f'{"Active" if label==0 else "Churned"}',
                color=colour, edgecolor='white')
    ax.set_title(col, fontsize=12)
    ax.legend(fontsize=9)
fig.suptitle('Feature Distributions by Churn Status', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3.4 Box plots for key features ---
key_features = ['tenure_months', 'customer_service_calls',
                'days_since_last_transaction', 'monthly_transactions']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, col in zip(axes, key_features):
    sns.boxplot(data=df, x='is_churned', y=col, ax=ax,
                palette=['#2196F3', '#F44336'], width=0.4)
    ax.set_xticklabels(['Active', 'Churned'])
    ax.set_title(col, fontsize=12)
fig.suptitle('Key Features by Churn Status', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Los box plots revelan patrones claros: los churners tienden a tener menor antiguedad, mas llamadas a servicio al cliente, mas dias desde su ultima transaccion y menos transacciones mensuales. Estas diferencias nos dan confianza en que un modelo de clasificacion va a encontrar senal suficiente para hacer predicciones utiles.

---
## 4. Ingenieria de features — Crear variables que capturen comportamiento

Las variables originales son utiles, pero a veces la senal esta en la combinacion de variables, no en cada una por separado. Vamos a crear tres features nuevas:

1. **transaction_trend** — Ratio entre dias sin transaccion y transacciones mensuales. Un valor alto indica que el cliente esta perdiendo engagement.
2. **balance_to_transaction_ratio** — Cuanto saldo tiene por cada transaccion. Ayuda a distinguir entre clientes que usan activamente la cuenta y los que solo tienen dinero estacionado.
3. **engagement_score** — Un score compuesto que combina numero de productos, frecuencia de transacciones, tarjeta de credito y cuenta de ahorros.

La razon por la que creo estas variables es que los modelos de ML trabajan con correlaciones lineales o arboles de decision — a veces necesitan que les "mastiquemos" la informacion en forma de ratios o scores compuestos para capturar relaciones que no son obvias en las variables crudas.

In [ ]:
# --- Derived features ---

# Transaction trend proxy: ratio of days since last txn to avg monthly txns
# Higher values indicate declining engagement
df['transaction_trend'] = (
    df['days_since_last_transaction'] / (df['monthly_transactions'] + 1)
).round(4)

# Balance-to-transaction ratio: how much balance per transaction
df['balance_to_transaction_ratio'] = (
    df['monthly_balance_avg'] / (df['avg_transaction_amount'] * df['monthly_transactions'] + 1)
).round(4)

# Engagement score: composite metric (higher = more engaged)
df['engagement_score'] = (
    0.3 * df['num_products'] / df['num_products'].max()
    + 0.3 * df['monthly_transactions'] / df['monthly_transactions'].max()
    + 0.2 * df['has_credit_card']
    + 0.2 * df['has_savings']
).round(4)

print('New features created:')
print(df[['transaction_trend', 'balance_to_transaction_ratio', 'engagement_score']].describe().round(3))

In [ ]:
# Quick sanity check: correlation of new features with churn
new_feats = ['transaction_trend', 'balance_to_transaction_ratio', 'engagement_score']
for f in new_feats:
    corr_val = df[f].corr(df['is_churned'])
    print(f'{f:>35s}  corr with churn = {corr_val:+.3f}')

La correlacion positiva de `transaction_trend` con churn confirma nuestra hipotesis: clientes que llevan mas dias sin transacciones relativo a su frecuencia habitual tienden a irse. El `engagement_score` tiene correlacion negativa, lo cual tambien tiene sentido — mas engagement, menos churn.

---
## 5. Pipeline de modelado — Comparar 3 algoritmos con SMOTE y validacion cruzada

Aqui es donde empieza lo interesante. Vamos a:

1. **Dividir los datos** en train (80%) y test (20%) con estratificacion para mantener la misma proporcion de churn en ambos conjuntos.
2. **Aplicar SMOTE** al conjunto de entrenamiento para balancear las clases. SMOTE crea ejemplos "sinteticos" de la clase minoritaria interpolando entre churners existentes. Lo aplicamos SOLO al train — nunca al test, porque el test debe reflejar la distribucion real.
3. **Estandarizar** las features para que todas esten en la misma escala.
4. **Entrenar 3 modelos** diferentes y compararlos con validacion cruzada de 5 folds.
5. **Hacer tuning de hiperparametros** al mejor modelo con GridSearchCV.

La razon por la que comparo tres modelos es que cada uno tiene fortalezas diferentes: Logistic Regression es interpretable y rapida, Random Forest es robusta y maneja bien features no lineales, y XGBoost generalmente da el mejor rendimiento en datos tabulares.

In [ ]:
# --- 5.1 Define X, y and split ---
feature_cols = [
    'age', 'tenure_months', 'monthly_transactions', 'avg_transaction_amount',
    'num_products', 'has_credit_card', 'has_savings', 'customer_service_calls',
    'days_since_last_transaction', 'monthly_balance_avg',
    'transaction_trend', 'balance_to_transaction_ratio', 'engagement_score'
]

X = df[feature_cols].copy()
y = df['is_churned'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(f'Train churn rate: {y_train.mean():.2%} | Test churn rate: {y_test.mean():.2%}')

In [ ]:
# --- 5.2 SMOTE for class imbalance ---
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'After SMOTE -> Train samples: {X_train_sm.shape[0]:,}')
print(f'Class distribution:\n{pd.Series(y_train_sm).value_counts()}')

In [ ]:
# --- 5.3 Scale features ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# --- 5.4 Define models ---
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        random_state=RANDOM_STATE, eval_metric='logloss',
        use_label_encoder=False, n_jobs=-1
    )
}

In [ ]:
# --- 5.5 Cross-validation (5-fold) ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train_sm,
                            cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:>22s}  |  AUC = {scores.mean():.4f} +/- {scores.std():.4f}')

In [ ]:
# --- 5.6 Train all models on full (SMOTE) training set ---
trained_models = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train_sm)
    trained_models[name] = model
    y_pred = model.predict(X_test_scaled)
    print(f'\n--- {name} ---')
    print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))

In [ ]:
# --- 5.7 Hyperparameter tuning for best CV model ---
# Identify best model from CV
best_model_name = max(cv_results, key=lambda k: cv_results[k].mean())
print(f'Best CV model: {best_model_name} -> tuning hyperparameters...\n')

param_grids = {
    'Logistic Regression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2']
    },
    'Random Forest': {
        'n_estimators': [100, 300],
        'max_depth': [8, 15, None],
        'min_samples_split': [2, 5]
    },
    'XGBoost': {
        'n_estimators': [100, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.05, 0.1]
    }
}

grid_search = GridSearchCV(
    models[best_model_name],
    param_grids[best_model_name],
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train_scaled, y_train_sm)

print(f'Best params: {grid_search.best_params_}')
print(f'Best CV AUC:  {grid_search.best_score_:.4f}')

best_model = grid_search.best_estimator_
trained_models[f'{best_model_name} (tuned)'] = best_model

---
## 6. Evaluacion — Que tan bien predecimos el churn

Ahora evaluamos los tres modelos en el conjunto de test (datos que nunca vieron durante el entrenamiento). Vamos a usar varias metricas porque en problemas de clasificacion desbalanceada la accuracy sola es enganosa — un modelo que prediga "activo" para todos tendria ~80% de accuracy pero seria inutil. Lo que nos importa mas aqui es el **recall** (que porcentaje de churners reales detectamos) y la **curva ROC/AUC** (que tan bien separa el modelo entre clientes activos y churners).

In [ ]:
# --- 6.1 Confusion matrices ---
eval_models = {k: v for k, v in trained_models.items() if '(tuned)' not in k}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (name, model) in zip(axes, eval_models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Active', 'Churned'],
                yticklabels=['Active', 'Churned'])
    ax.set_title(name, fontsize=12)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
fig.suptitle('Confusion Matrices', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- 6.2 ROC Curves (all 3 models on one plot) ---
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#2196F3', '#4CAF50', '#FF9800']

auc_scores = {}
for (name, model), color in zip(eval_models.items(), colors):
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    auc_scores[name] = roc_auc
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - Model Comparison', fontsize=14)
ax.legend(fontsize=11, loc='lower right')
plt.tight_layout()
plt.show()

print('\nAUC Comparison:')
for name, score in sorted(auc_scores.items(), key=lambda x: -x[1]):
    print(f'  {name:>22s}: {score:.4f}')

In [ ]:
# --- 6.3 Precision-Recall Curve ---
fig, ax = plt.subplots(figsize=(8, 6))

for (name, model), color in zip(eval_models.items(), colors):
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = auc(rec, prec)
    ax.plot(rec, prec, color=color, lw=2, label=f'{name} (PR-AUC = {pr_auc:.3f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# --- 6.4 Feature importance (best base model) ---
# Use XGBoost or Random Forest for feature importance
fi_model_name = 'XGBoost' if 'XGBoost' in eval_models else 'Random Forest'
fi_model = eval_models[fi_model_name]

importances = fi_model.feature_importances_
feat_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(feat_imp['feature'], feat_imp['importance'], color='#2196F3', edgecolor='white')
ax.set_xlabel('Importance', fontsize=12)
ax.set_title(f'Feature Importance ({fi_model_name})', fontsize=14)
plt.tight_layout()
plt.show()

El grafico de importancia de features confirma lo que vimos en la exploracion: las llamadas a servicio al cliente, los dias desde la ultima transaccion y la antiguedad son los predictores mas fuertes de churn. Esto tiene sentido intuitivo — un cliente que llama mucho al soporte esta frustrado, uno que no transacciona se esta desconectando, y los clientes nuevos aun no tienen lealtad. Las features que creamos (como `transaction_trend`) tambien aparecen con importancia relevante, lo cual valida el esfuerzo de ingenieria de features.

---
## 7. Optimizacion del threshold — El 0.5 por defecto casi nunca es el mejor

Este es un paso que mucha gente se salta, pero marca una diferencia enorme en la practica. Por defecto, los clasificadores usan 0.5 como umbral de decision: si la probabilidad predicha es mayor a 0.5, clasifica como churner. Pero en nuestro caso, perder un churner (falso negativo) es mucho mas costoso que contactar innecesariamente a un cliente activo (falso positivo). Por eso vamos a probar diferentes thresholds y encontrar el que maximiza el F1 score, que balancea precision y recall.

In [ ]:
# Use the best performing model for threshold analysis
final_model = best_model
y_proba_final = final_model.predict_proba(X_test_scaled)[:, 1]

# Precision-recall tradeoff across thresholds
thresholds = np.arange(0.1, 0.9, 0.01)
results = []
for t in thresholds:
    y_pred_t = (y_proba_final >= t).astype(int)
    results.append({
        'threshold': t,
        'precision': precision_score(y_test, y_pred_t, zero_division=0),
        'recall': recall_score(y_test, y_pred_t, zero_division=0),
        'f1': f1_score(y_test, y_pred_t, zero_division=0)
    })

thresh_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresh_df['threshold'], thresh_df['precision'], label='Precision',
        color='#2196F3', lw=2)
ax.plot(thresh_df['threshold'], thresh_df['recall'], label='Recall',
        color='#F44336', lw=2)
ax.plot(thresh_df['threshold'], thresh_df['f1'], label='F1 Score',
        color='#4CAF50', lw=2, linestyle='--')

# Optimal threshold (max F1)
optimal_idx = thresh_df['f1'].idxmax()
optimal_thresh = thresh_df.loc[optimal_idx, 'threshold']
ax.axvline(optimal_thresh, color='gray', linestyle=':', lw=1.5,
           label=f'Optimal threshold = {optimal_thresh:.2f}')

ax.set_xlabel('Decision Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Precision-Recall Tradeoff by Threshold', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'\nOptimal threshold (max F1): {optimal_thresh:.2f}')
print(f'  Precision = {thresh_df.loc[optimal_idx, "precision"]:.3f}')
print(f'  Recall    = {thresh_df.loc[optimal_idx, "recall"]:.3f}')
print(f'  F1        = {thresh_df.loc[optimal_idx, "f1"]:.3f}')

In [ ]:
# Final classification report with optimised threshold
y_pred_optimal = (y_proba_final >= optimal_thresh).astype(int)

print(f'Classification Report (threshold = {optimal_thresh:.2f}):')
print(classification_report(y_test, y_pred_optimal, target_names=['Active', 'Churned']))

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred_optimal)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Active', 'Churned'],
            yticklabels=['Active', 'Churned'])
ax.set_title(f'Confusion Matrix (threshold = {optimal_thresh:.2f})', fontsize=12)
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

---
## 8. Impacto de negocio — Traducir el modelo a dolares

Un modelo de ML no sirve si no podemos explicar su valor en terminos que el negocio entienda. Vamos a calcular cuanto dinero ahorra este modelo comparado con no hacer nada. Las suposiciones son conservadoras:

| Supuesto | Valor |
|----------|-------|
| Ingreso mensual promedio por cliente | $85 |
| Vida promedio del cliente (meses) | 24 |
| Costo de adquisicion de cliente (CAC) | $250 |
| Costo de campana de retencion por cliente | $15 |
| Tasa de exito de retencion con intervencion | 30% |

**Sin el modelo:** el banco pierde todos los clientes que se van sin hacer nada proactivo.

**Con el modelo:** los clientes flaggeados reciben una oferta de retencion. Con una tasa de exito del 30%, salvamos una porcion significativa.

Lo importante: cada cliente salvado vale ~$2,040 en LTV restante, y contactar a un cliente que no iba a irse solo cuesta $15. El costo de un falso positivo es muy bajo comparado con el beneficio de un verdadero positivo.

In [ ]:
# --- Quantified business impact ---
avg_monthly_revenue = 85
avg_lifetime_months = 24
ltv = avg_monthly_revenue * avg_lifetime_months  # $2,040
campaign_cost = 15
retention_rate = 0.30

tp = cm[1, 1]  # true positives
fp = cm[0, 1]  # false positives
fn = cm[1, 0]  # false negatives

customers_saved = tp * retention_rate
revenue_saved = customers_saved * ltv
campaign_total_cost = (tp + fp) * campaign_cost
revenue_lost = fn * ltv
net_benefit = revenue_saved - campaign_total_cost

# Scale to full customer base (8000)
scale_factor = 8000 / len(y_test)

print('=== Business Impact (scaled to 8,000 customers) ===')
print(f'  Churners identified (TP):      {tp * scale_factor:,.0f}')
print(f'  Customers saved (30% rate):     {customers_saved * scale_factor:,.0f}')
print(f'  Revenue saved:                 ${revenue_saved * scale_factor:,.0f}')
print(f'  Campaign cost:                 ${campaign_total_cost * scale_factor:,.0f}')
print(f'  Missed churners (FN):           {fn * scale_factor:,.0f}')
print(f'  Revenue lost (missed):         ${revenue_lost * scale_factor:,.0f}')
print(f'  ----------------------------------------')
print(f'  NET BENEFIT:                   ${net_benefit * scale_factor:,.0f}')

---
## 9. Conclusiones

1. **Mejor modelo:** El modelo tuneado logro un poder discriminatorio fuerte (ver AUC arriba), confirmando que las senales de comportamiento del cliente — particularmente `customer_service_calls`, `days_since_last_transaction` y `tenure_months` — son predictores confiables de churn.

2. **SMOTE resolvio el desbalance de clases:** Sin sobremuestreo, los modelos tendian a sub-predecir la clase minoritaria. SMOTE mejoro el recall en churners sin una perdida excesiva de precision.

3. **La optimizacion del threshold importa:** Mover el umbral de decision del 0.50 por defecto al valor optimo de F1 mejoro el balance precision-recall para el caso de uso de negocio, donde los falsos negativos (churners no detectados) son mas costosos que los falsos positivos.

4. **La ingenieria de features agrego valor:** Variables derivadas como `transaction_trend` y `engagement_score` capturaron matices de comportamiento que las variables crudas por si solas no capturaban.

5. **El impacto de negocio es tangible:** Incluso con suposiciones conservadoras (30% de tasa de retencion), la deteccion temprana genera ahorros netos significativos. El modelo esta listo para A/B testing en produccion.

### Proximos pasos

- Integrar datos transaccionales reales y validar en un periodo de tiempo holdout.
- Explorar enfoques de deep learning (por ejemplo, LSTM sobre secuencias de transacciones).
- Desplegar como un API de scoring con inferencia batch programada y dashboards de monitoreo.